# **Machine Learning Final Project - News Source Classification Pt. 1**
*Tonoya Ahmed, Hanna Pucheu, Angela Tan*

# Part 1: Data Preprocessing


In [ ]:
!pip install feedparser
!pip install vaderSentiment

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, precision_recall_fscore_support

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 2.7 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=9bb72f4a59446b8d8c97d6bcb1902faead6d465ec4eea88c381111a6fbdebed7
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built sgmllib3k
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 3.2 MB/s eta 0:00:00
ERROR: Operation cancelled by user


In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np

In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1a. Cleaning and Preprocessing Functions

In [ ]:
%%writefile preprocess.py
# FINAL PREPROCESSING FUNCTION, ready for leaderboard submission
# imports
import warnings
warnings.filterwarnings("ignore")
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

import torch
import transformers
import pandas as pd
import numpy as np
#from google.colab import drive
import feedparser
import pandas as pd

import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import spacy
nlp = spacy.load("en_core_web_sm")
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, precision_recall_fscore_support

# feature engineering functions

# HEADLINE CLEANING
def clean_headline(headline):
    if pd.isna(headline):
        return ""
    headline = str(headline).lower() # lower case
    headline = re.sub(r"\s*\|\s*.*$", "", headline) # might be able to remove this
    headline = re.sub(r"\s+", " ", headline).strip()
    return headline

import spacy
nlp = spacy.load("en_core_web_sm")

def lemmatize_headline(text):
    doc = nlp(text)
    return " ".join([token.lemma_ for token in doc])


# SENTIMENT ANALYZING
analyzer = SentimentIntensityAnalyzer()
def get_sentiment(text):
    return analyzer.polarity_scores(text)["compound"]
# label sentiment
def label_sentiment(score):
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    else:
        return "neutral"

# ADJECTIVE RATIO
def adjective_ratio(text):
    doc = nlp(text)
    words = [token for token in doc if token.is_alpha]
    if len(words) == 0:
        return 0
    adj_count = sum(1 for token in words if token.pos_ == "ADJ")
    return adj_count / len(words)

def headline_feature_engineering(df): # input should be [url, headline]
  # headline word length
  df["headline_length"] = df["headline"].str.split().apply(len)

  # headline average word length
  df["avg_word_length"] = df["headline"].apply(
    lambda x: sum(len(w) for w in x.split()) / len(x.split()) if len(x.split()) > 0 else 0)

  # adjective ratio
  df["adj_ratio"] = df["headline"].apply(adjective_ratio)

  # sentiment
  df["sentiment"] = df["headline"].apply(get_sentiment)
  df["sentiment_label"] = df["sentiment"].apply(label_sentiment)

  # has question?
  df["has_question"] = df["headline"].str.contains(r"\?").astype(int)

  return df


def prepare_data(csv_path):

  df = pd.read_csv(csv_path)

  # drop any rows that don't contain foxnews.com or nbcnews.com
 #  df = df[df["url"].str.contains("foxnews.com|nbcnews.com", na=False)].copy()
  # apply label: 1 if fox news, 0 if nbc
  df["label"] = np.where(df["url"].str.contains("foxnews.com"), 1, 0)

  # clean headline
  df["headline"] = df["headline"].apply(clean_headline)
  df["headline"] = df["headline"].apply(lemmatize_headline)

  # apply features
  df = headline_feature_engineering(df)

  X = df.drop(columns=["label"])
  y = df["label"]
  return X, y

Overwriting preprocess.py


## 1b. Data

In [ ]:
# original

from preprocess import prepare_data

path = "/content/drive/MyDrive/ML Project/og_headlines_output.csv"
X, y = prepare_data(path) # test on our function


In [ ]:
news_df = pd.concat([X,y], axis=1)

In [ ]:
from preprocess import prepare_data
X_sample,y_sample = prepare_data("/content/drive/MyDrive/ML Project/new_dataset.csv")
sampled_new_data = pd.concat([X_sample,y_sample], axis=1)
sampled_new_data = (sampled_new_data.groupby("label", group_keys=False).apply(lambda x: x.sample(n=5500, random_state=42)))
sampled_new_data = sampled_new_data.reset_index(drop=True)

In [ ]:
#join the data
news_df_cleaned = pd.concat([news_df, sampled_new_data])
news_df_cleaned.shape

(14805, 9)

# Part 2: Modeling

## 2a. Train test split and Vectorize TF-IDF

In [ ]:
# Text data
X_text = news_df_cleaned["headline"]

# Numeric data
X_numeric = news_df_cleaned[[
    "sentiment",
    "headline_length",
    "avg_word_length",
    "adj_ratio",
    "has_question"
]]

y = news_df_cleaned["label"]

In [ ]:
from sklearn.model_selection import train_test_split
# Train test split
X_text_train, X_text_test, X_num_train, X_num_test, y_train, y_test = train_test_split(
    X_text, X_numeric, y,
    test_size=0.2,
    random_state=42
)

In [ ]:
# Scaled features
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_num_train_scaled = scaler.fit_transform(X_num_train)
X_num_test_scaled = scaler.transform(X_num_test)

In [ ]:
# # TF-IDF

from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC


#grid search to find best n-grams range
pipe = Pipeline([
    ("features", FeatureUnion([
        ("word_tfidf", TfidfVectorizer(
            max_features=10000,
            min_df=2,
            sublinear_tf=True,
            stop_words="english"
        )),
        ("char_tfidf", TfidfVectorizer(
            analyzer="char_wb",
            max_features=5000
        ))
    ])),
    ("model", LinearSVC(C=0.5))
])

param_grid = {
    "features__word_tfidf__ngram_range": [(1, 1), (1, 2), (1, 3)],
    "features__char_tfidf__ngram_range": [(2, 4), (3, 5), (4, 6)]
}

grid = GridSearchCV(
    pipe,
    param_grid=param_grid,
    scoring="f1_weighted",
    cv=3,
    n_jobs=-1
)

grid.fit(X_text_train, y_train)

best_word_ngram = grid.best_params_["features__word_tfidf__ngram_range"]
best_char_ngram = grid.best_params_["features__char_tfidf__ngram_range"]

vectorizer = TfidfVectorizer(
    ngram_range=best_word_ngram,
    max_features=10000,
    min_df=2,
    sublinear_tf=True,
    stop_words="english"
)

char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=best_char_ngram,
    max_features=5000
)

X_train_tfidf = vectorizer.fit_transform(X_text_train)
X_test_tfidf = vectorizer.transform(X_text_test)

X_char_train = char_vectorizer.fit_transform(X_text_train)
X_char_test = char_vectorizer.transform(X_text_test)

In [ ]:
# Combine TF-IDF with engineered features
from scipy.sparse import hstack

X_train_final = hstack([
    X_train_tfidf,
    X_char_train,
    X_num_train_scaled
])

X_test_final = hstack([
    X_test_tfidf,
    X_char_test,
    X_num_test_scaled
])

# Combined with scaled

X_train_scaled = hstack([X_train_tfidf, X_char_train, X_num_train_scaled])
X_test_scaled = hstack([X_test_tfidf, X_char_test, X_num_test_scaled])

## 2b. Baseline Models

### i. LogReg Baseline Model (from project guidelines)

In [ ]:
# Create variables to store evaluation metrics

eval_metrics = pd.DataFrame(columns=['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score'])

In [ ]:
# Baseline model on just the TF-IDF
model = LogisticRegression(max_iter=100)
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print("Classification Report:\n", classification_report(y_test, y_pred))



new_row= {
    'Model': 'LogRegWTFIDF',
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': classification_report(y_test, y_pred, output_dict=True)['weighted avg']['precision'],
    'Recall': classification_report(y_test, y_pred, output_dict=True)['weighted avg']['recall'],
    'F1-Score': classification_report(y_test, y_pred, output_dict=True)['weighted avg']['f1-score']
}
eval_metrics = pd.concat([eval_metrics, pd.DataFrame([new_row])])

Accuracy: 0.7899
Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.78      0.79      1484
           1       0.78      0.80      0.79      1477

    accuracy                           0.79      2961
   macro avg       0.79      0.79      0.79      2961
weighted avg       0.79      0.79      0.79      2961



### ii. LogReg  Model with new features

In [ ]:
# Baseline model on the TF-IDF combined with new features
model = LogisticRegression(max_iter=100)
model.fit(X_train_final, y_train)

y_pred = model.predict(X_test_final)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print("Classification Report:\n", classification_report(y_test, y_pred))



new_row= {
    'Model': 'LogRegWBoth',
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': classification_report(y_test, y_pred, output_dict=True)['weighted avg']['precision'],
    'Recall': classification_report(y_test, y_pred, output_dict=True)['weighted avg']['recall'],
    'F1-Score': classification_report(y_test, y_pred, output_dict=True)['weighted avg']['f1-score']
}
eval_metrics = pd.concat([eval_metrics, pd.DataFrame([new_row])])

Accuracy: 0.8673
Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.86      0.87      1484
           1       0.86      0.87      0.87      1477

    accuracy                           0.87      2961
   macro avg       0.87      0.87      0.87      2961
weighted avg       0.87      0.87      0.87      2961



### iii. RFC

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model_rfc = RandomForestClassifier()
model_rfc.fit(X_train_final, y_train)
preds = model_rfc.predict(X_test_final)

print("Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))

new_row= {
    'Model': 'RandomForestClassifier',
    'Accuracy': accuracy_score(y_test, preds),
    'Precision': classification_report(y_test, preds, output_dict=True)['weighted avg']['precision'],
    'Recall': classification_report(y_test, preds, output_dict=True)['weighted avg']['recall'],
    'F1-Score': classification_report(y_test, preds, output_dict=True)['weighted avg']['f1-score']
}
eval_metrics = pd.concat([eval_metrics, pd.DataFrame([new_row])])

Accuracy: 0.8595069233367105
              precision    recall  f1-score   support

           0       0.83      0.91      0.87      1484
           1       0.90      0.81      0.85      1477

    accuracy                           0.86      2961
   macro avg       0.86      0.86      0.86      2961
weighted avg       0.86      0.86      0.86      2961



### iv. Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model_dt = DecisionTreeClassifier()
model_dt.fit(X_train_final, y_train)
preds = model_dt.predict(X_test_final)

print("Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))

new_row= {
    'Model': 'DecisionTreeClassifier',
    'Accuracy': accuracy_score(y_test, preds),
    'Precision': classification_report(y_test, preds, output_dict=True)['weighted avg']['precision'],
    'Recall': classification_report(y_test, preds, output_dict=True)['weighted avg']['recall'],
    'F1-Score': classification_report(y_test, preds, output_dict=True)['weighted avg']['f1-score']
}
eval_metrics = pd.concat([eval_metrics, pd.DataFrame([new_row])])

Accuracy: 0.7858831475852752
              precision    recall  f1-score   support

           0       0.79      0.79      0.79      1484
           1       0.79      0.78      0.78      1477

    accuracy                           0.79      2961
   macro avg       0.79      0.79      0.79      2961
weighted avg       0.79      0.79      0.79      2961



### v. Kernel SVM

In [ ]:
from sklearn.svm import SVC
# OTHER kernels: 'linear', 'poly', 'rbf', 'sigmoid'
model_kernelsvm = SVC(kernel='rbf')
model_kernelsvm.fit(X_train_final, y_train)
preds = model_kernelsvm.predict(X_test_final)

print("Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))


new_row= {
    'Model': 'KernelSVC',
    'Accuracy': accuracy_score(y_test, preds),
    'Precision': classification_report(y_test, preds, output_dict=True)['weighted avg']['precision'],
    'Recall': classification_report(y_test, preds, output_dict=True)['weighted avg']['recall'],
    'F1-Score': classification_report(y_test, preds, output_dict=True)['weighted avg']['f1-score']
}
eval_metrics = pd.concat([eval_metrics, pd.DataFrame([new_row])])

Accuracy: 0.8611955420466059
              precision    recall  f1-score   support

           0       0.87      0.86      0.86      1484
           1       0.86      0.87      0.86      1477

    accuracy                           0.86      2961
   macro avg       0.86      0.86      0.86      2961
weighted avg       0.86      0.86      0.86      2961



### vi. Linear SVM

In [ ]:
from sklearn.svm import LinearSVC

model_linearsvc = LinearSVC(C = 0.5)
model_linearsvc.fit(X_train_final, y_train)

y_pred = model_linearsvc.predict(X_test_final)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


new_row= {
    'Model': 'LinearSVC',
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': classification_report(y_test, y_pred, output_dict=True)['weighted avg']['precision'],
    'Recall': classification_report(y_test, y_pred, output_dict=True)['weighted avg']['recall'],
    'F1-Score': classification_report(y_test, y_pred, output_dict=True)['weighted avg']['f1-score']
}
eval_metrics = pd.concat([eval_metrics, pd.DataFrame([new_row])])

Accuracy: 0.8713272543059777
              precision    recall  f1-score   support

           0       0.88      0.87      0.87      1484
           1       0.87      0.88      0.87      1477

    accuracy                           0.87      2961
   macro avg       0.87      0.87      0.87      2961
weighted avg       0.87      0.87      0.87      2961



### vii. KNN

In [ ]:
X_numeric = news_df_cleaned[[
    "sentiment",
    "headline_length",
    "avg_word_length",
    "adj_ratio",
    "has_question"
]]

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
# knn model
# run on scaled data
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

y_pred = knn.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

new_row= {
    'Model': 'KNeighborsClassifier',
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': classification_report(y_test, y_pred, output_dict=True)['weighted avg']['precision'],
    'Recall': classification_report(y_test, y_pred, output_dict=True)['weighted avg']['recall'],
    'F1-Score': classification_report(y_test, y_pred, output_dict=True)['weighted avg']['f1-score']
}
eval_metrics = pd.concat([eval_metrics, pd.DataFrame([new_row])])

Accuracy: 0.7426545086119554
              precision    recall  f1-score   support

           0       0.74      0.74      0.74      1484
           1       0.74      0.74      0.74      1477

    accuracy                           0.74      2961
   macro avg       0.74      0.74      0.74      2961
weighted avg       0.74      0.74      0.74      2961



notes: KNN is not ideal for TF-IDF since the data is very high dimensional and sparse. KNN becomes noisy with high dimensional data.

### vii. Neural Networks

In [ ]:
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(
    hidden_layer_sizes=(100, 50),
    activation="relu",
    max_iter=20,
    random_state=42
)

mlp.fit(X_train_scaled, y_train)

y_pred = mlp.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

new_row= {
    'Model': 'NeuralNetwork',
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': classification_report(y_test, y_pred, output_dict=True)['weighted avg']['precision'],
    'Recall': classification_report(y_test, y_pred, output_dict=True)['weighted avg']['recall'],
    'F1-Score': classification_report(y_test, y_pred, output_dict=True)['weighted avg']['f1-score']
}
eval_metrics = pd.concat([eval_metrics, pd.DataFrame([new_row])])

Accuracy: 0.8689631881121243
              precision    recall  f1-score   support

           0       0.87      0.86      0.87      1484
           1       0.86      0.88      0.87      1477

    accuracy                           0.87      2961
   macro avg       0.87      0.87      0.87      2961
weighted avg       0.87      0.87      0.87      2961



## 2c. Evaluation

In [ ]:
eval_metrics

,Model,Accuracy,Precision,Recall,F1-Score
0,LogRegWTFIDF,0.789936,0.790049,0.789936,0.789922
0,LogRegWBoth,0.867275,0.867368,0.867275,0.867269
0,RandomForestClassifier,0.859507,0.863203,0.859507,0.859128
0,DecisionTreeClassifier,0.785883,0.785886,0.785883,0.785881
0,KernelSVC,0.861196,0.861236,0.861196,0.861194
0,LinearSVC,0.871327,0.871380,0.871327,0.871325
0,KNeighborsClassifier,0.742655,0.742654,0.742655,0.742654
0,NeuralNetwork,0.868963,0.869049,0.868963,0.868958


## 2d. Tuning Models

### i. Tuning LogReg

In [ ]:
#take a while to run

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

param_grid = [
    {
        "solver": ["lbfgs", "liblinear", "saga"],
        "penalty": ["l2"],
        "C": [0.1, 1, 10]

    },
    {
        "solver": ["liblinear", "saga"],
        "penalty": ["l1"],
        "C": [0.1, 1, 10],
    }
]

grid = grid = GridSearchCV(
    LogisticRegression(max_iter=100),
    param_grid=param_grid,
    scoring="accuracy",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train_final, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV Accuracy:", grid.best_score_)

best_logreg = grid.best_estimator_

y_pred = best_logreg.predict(X_test_final)

accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, output_dict=True)

print(f"Accuracy: {accuracy:.4f}")
print("Classification Report:\n", classification_report(y_test, y_pred))

new_row = {
    "Model": "TunedLogRegWBoth",
    "Accuracy": accuracy,
    "Precision": report["weighted avg"]["precision"],
    "Recall": report["weighted avg"]["recall"],
    "F1-Score": report["weighted avg"]["f1-score"]
}

eval_metrics = pd.concat(
    [eval_metrics, pd.DataFrame([new_row])],
    ignore_index=True
)

eval_metrics

Fitting 5 folds for each of 15 candidates, totalling 75 fits
Best parameters: {'C': 10, 'penalty': 'l2', 'solver': 'lbfgs'}
Best CV Accuracy: 0.8767303671865194
Accuracy: 0.8720
Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.87      0.87      1484
           1       0.87      0.87      0.87      1477

    accuracy                           0.87      2961
   macro avg       0.87      0.87      0.87      2961
weighted avg       0.87      0.87      0.87      2961



,Model,Accuracy,Precision,Recall,F1-Score
0,LogRegWTFIDF,0.789936,0.790049,0.789936,0.789922
1,LogRegWBoth,0.867275,0.867368,0.867275,0.867269
2,RandomForestClassifier,0.859507,0.863203,0.859507,0.859128
3,DecisionTreeClassifier,0.785883,0.785886,0.785883,0.785881
4,KernelSVC,0.861196,0.861236,0.861196,0.861194
5,LinearSVC,0.871327,0.871380,0.871327,0.871325
6,KNeighborsClassifier,0.742655,0.742654,0.742655,0.742654
7,NeuralNetwork,0.868963,0.869049,0.868963,0.868958
8,TunedLogRegWBoth,0.872003,0.872003,0.872003,0.872003


In [ ]:
print("Best CV Accuracy:", grid.best_score_)

Best CV Accuracy: 0.8767303671865194


In [ ]:
# final evaluation of tuned model

accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, output_dict=True)

print(f"Accuracy: {accuracy:.4f}")
print("Classification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.8720
Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.87      0.87      1484
           1       0.87      0.87      0.87      1477

    accuracy                           0.87      2961
   macro avg       0.87      0.87      0.87      2961
weighted avg       0.87      0.87      0.87      2961



### ii. Checking feature importance for LogReg

In [ ]:

coefs = np.abs(best_logreg.coef_[0])

# sizes
n_word = len(vectorizer.get_feature_names_out())
n_char = len(char_vectorizer.get_feature_names_out())
n_extra = 5

# split
word_imp = coefs[:n_word].mean()
char_imp = coefs[n_word:n_word+n_char].mean()
extra_imp = coefs[n_word+n_char:].mean()

print("Word TF-IDF importance:", word_imp)
print("Char TF-IDF importance:", char_imp)
print("Engineered features importance:", extra_imp)

Word TF-IDF importance: 0.7287615755299719
Char TF-IDF importance: 0.6652531993129741
Engineered features importance: 0.38383188679875174


In [ ]:

coefs = np.abs(best_logreg.coef_[0])

# sizes
n_word = len(vectorizer.get_feature_names_out())
n_char = len(char_vectorizer.get_feature_names_out())

# engineered feature names
extra_names = ["headline_length", "avg_word_length", "adj_ratio", "sentiment", "has_question"]

# extract engineered feature coefficients
extra_coefs = coefs[n_word + n_char:]

# print importance
for name, val in zip(extra_names, extra_coefs):
    print(f"{name}: {val:.6f}")

headline_length: 0.038933
avg_word_length: 0.969610
adj_ratio: 0.840356
sentiment: 0.011207
has_question: 0.059054
